# Hard-core bosons with local loss

This executed tutorial follows site occupations $\langle n_i\rangle$ in a one-dimensional lattice. Occupied/empty hard-core-boson states are the ITensor spin states `Up`/`Dn`, with $n_i=S_i^z+1/2$. The bath removes particles through $L_i=\sqrt{\gamma_i}b_i$. We check an analytic one-site limit, closed-chain particle conservation, and lossy trajectory averages against a dense Lindblad calculation.


## Setup: initial particles, hopping, interaction, and loss

`initial=[1,1,0]` places two bosons on the left. `hopping=1` exchanges a boson with a neighboring vacancy, `interaction=0.7` changes the energy of neighboring occupied sites, and `loss_rates[2]=0.4` makes the middle site lossy. Time is measured in inverse energy units with $\hbar=1$.


In [1]:
# Two occupied sites start next to each other; only site 2 couples to the loss bath.
using HardcoreBosonChainLoss, Random, Printf, LinearAlgebra
model = Model(3; hopping=1, interaction=0.7, loss_rates=[0,0.4,0])
initial = [1,1,0]
println("MPS sites: ", length(model.sites), "; initial occupations: ", initial)

ArgumentError: ArgumentError: Package HardcoreBosonChainLoss not found in current path.
- Run `import Pkg; Pkg.add("HardcoreBosonChainLoss")` to install the HardcoreBosonChainLoss package.

## Analytic limit: one particle can only survive or be lost

With hopping absent, one occupied site obeys $\langle n(t)\rangle=e^{-\gamma t}$. The dense result below checks the physical meaning and rate convention of `loss_rates`.


In [2]:
# For one initially occupied site, the occupation is its survival probability.
isolated = Model(1; hopping=0, loss_rates=[0.8])
t = 0.7
exact = dense_evolve(isolated, [1], t).local_occupation[1]
@printf("one-site occupation %.9f; analytic %.9f\n", exact, exp(-0.8t))

UndefVarError: UndefVarError: `Model` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Coherent MPS check: no bath, conserved particle number

Setting all loss rates to zero leaves Hamiltonian motion. The two particles can redistribute, but $\sum_i\langle n_i\rangle=2$. The dense calculation checks site occupations independently of the MPS bond-gate split.


In [3]:
# With γ=0, the total occupation is the conserved particle number.
closed = Model(3; hopping=1, interaction=0.7, loss_rates=zeros(3))
mps = trajectory(closed, initial, 0.6; dt=0.01, maxdim=16, rng=MersenneTwister(1))
dense = dense_evolve(closed, initial, 0.6)
@printf("closed-chain max occupation error %.3e; particle count %.9f\n", maximum(abs.(mps.local_occupation-dense.local_occupation)), sum(mps.local_occupation))

UndefVarError: UndefVarError: `Model` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Lossy ensemble: trajectories reconstruct a density-matrix mean

Each MPS trajectory is one possible measurement record of particle losses. `mean_occupation` estimates $\mathrm{Tr}(n_i\rho)$; `se_occupation` is Monte Carlo sampling uncertainty. `mean_jumps` counts detected loss events. The dense Lindblad state `reference.rho` averages over all records without sampling. Finite time-step and MPS truncation errors are separate from the reported standard errors.


In [4]:
# Compare the ensemble estimate of Tr(nᵢρ) with the exact Lindblad value.
sampled = ensemble(model, initial, 0.3, 400; dt=0.02, maxdim=16, rng=MersenneTwister(3))
reference = dense_evolve(model, initial, 0.3)
for i in eachindex(initial)
    @printf("site %d: MPS %.6f ± %.6f; dense %.6f\n", i, sampled.mean_occupation[i], sampled.se_occupation[i], reference.local_occupation[i])
end
@printf("mean jumps %.5f; density-matrix trace %.9f\n", sampled.mean_jumps, real(tr(reference.rho)))

UndefVarError: UndefVarError: `MersenneTwister` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name also exists in Random.